In [ ]:
%pip install awswrangler scikit-learn joblib

In [ ]:
import awswrangler as wr

df = wr.athena.read_sql_query(
    sql='SELECT * FROM "db_rues"."ml_dataset_renovacion" LIMIT 50000',
    database="db_rues",
    workgroup="primary",
    ctas_approach=False,
    s3_output="s3://<S3_BUCKET>/athena-results/"
)

df.shape, df.head()

In [ ]:
df["renovo"].value_counts(dropna=False)

In [ ]:
df.info()

In [ ]:
target = "renovo"

categorical = [
    "codigo_camara",
    "camara_comercio",
    "tipo_sociedad",
    "organizacion_juridica",
    "categoria_matricula",
    "actividad_economica",
    "segmento_antiguedad",
]

numeric = [
    "tipo_persona",
    "antiguedad_empresa",
    "ultimo_ano_renovado",
    "anos_sin_renovar",
]

features = categorical + numeric

In [ ]:
import numpy as np
import pandas as pd

df_model = df[features + [target]].copy()
df_model = df_model.replace({pd.NA: np.nan})
df_model = df_model.dropna(subset=[target])
df_model[target] = df_model[target].astype(int)

for col in categorical:
    df_model[col] = df_model[col].astype("object")
    df_model[col] = df_model[col].where(df_model[col].notna(), np.nan)

for col in numeric:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

X = df_model[features]
y = df_model[target]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(transformers=[
    ("cat", categorical_pipeline, categorical),
    ("num", numeric_pipeline, numeric),
])

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import json
import joblib
from pathlib import Path
from datetime import datetime

Path("artifacts").mkdir(exist_ok=True)

joblib.dump(model, "artifacts/model.joblib")

metrics = {
    "run_at": datetime.now().isoformat(timespec="seconds"),
    "rows_total": int(len(df_model)),
    "rows_train": int(len(X_train)),
    "rows_test": int(len(X_test)),
    "target_distribution": {
        str(k): int(v) for k, v in y.value_counts().sort_index().items()
    },
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "classification_report": classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    ),
}

with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open("artifacts/columns.json", "w") as f:
    json.dump({
        "target": target,
        "categorical_features": categorical,
        "numeric_features": numeric,
    }, f, indent=2)

list(Path("artifacts").iterdir())

In [ ]:
import boto3

bucket = "<S3_BUCKET>"
prefix = "ml/artifacts/renovacion"

s3 = boto3.client("s3", region_name="us-east-1")

s3.upload_file("artifacts/model.joblib", bucket, f"{prefix}/model.joblib")
s3.upload_file("artifacts/metrics.json", bucket, f"{prefix}/metrics.json")
s3.upload_file("artifacts/columns.json", bucket, f"{prefix}/columns.json")

print(f"s3://{bucket}/{prefix}/")